In [1]:
import numpy as np

# Définition du Line World sous la forme d'un Markov Decision Process

In [2]:
LINE_WORLD_LENGTH = 5

In [3]:
A = np.array([0, 1]) # 0: Gauche, 1: Droite

In [4]:
S = np.array([i for i in range(LINE_WORLD_LENGTH)]) # Agent position

In [5]:
R = np.array([0, -1, 1]) # Récompenses

In [6]:
p = np.zeros((len(S), len(A), len(S), len(R)))

p[1, 0, 0, 1] = 1.0
p[LINE_WORLD_LENGTH - 2, 1, LINE_WORLD_LENGTH - 1, 2] = 1.0

for s in range(2, LINE_WORLD_LENGTH - 1):
  p[s, 0, s - 1, 0] = 1.0

for s in range(1, LINE_WORLD_LENGTH - 2):
  p[s, 1, s + 1, 0] = 1.0

In [7]:
T = np.array([0, LINE_WORLD_LENGTH - 1]) # terminal states

# Policy Evaluation

In [8]:
def iterative_policy_evaluation(
    model_S: np.ndarray,
    model_A: np.ndarray,
    model_R: np.ndarray,
    model_p: np.ndarray,
    model_T: np.ndarray,
    pi: np.ndarray,
    gamma: float = 0.999999,
    theta: float = 0.000001):
  V = np.random.random(len(model_S))
  V[model_T] = 0.0
  while True:
    delta = 0
    for s_index in range(len(model_S)):
      v = V[model_S[s_index]]
      total = 0.0
      for a_index in range(len(model_A)):
        action_total = 0.0
        for s_p_index in range(len(model_S)):
          for r_index in range(len(model_R)):
            action_total += model_p[s_index, a_index, s_p_index, r_index] * (model_R[r_index] + gamma * V[s_p_index])
        total += pi[s_index, a_index] * action_total
      V[model_S[s_index]] = total
      delta = np.maximum(delta, np.abs((total - v)))
    if delta < theta:
      break
  return V

In [9]:
pi_right = np.zeros((len(S), len(A)))
pi_right[:, 1] = 1.0

In [10]:
iterative_policy_evaluation(S, A, R, p, T, pi_right)

array([0.      , 0.999998, 0.999999, 1.      , 0.      ])

In [11]:
pi_left = np.zeros((len(S), len(A)))
pi_left[:, 0] = 1.0

In [12]:
iterative_policy_evaluation(S, A, R, p, T, pi_left)

array([ 0.      , -1.      , -0.999999, -0.999998,  0.      ])

In [13]:
pi_uniform_random = np.zeros((len(S), len(A)))
pi_uniform_random[:, 0] = 0.5
pi_uniform_random[:, 1] = 0.5

In [14]:
iterative_policy_evaluation(S, A, R, p, T, pi_uniform_random)

array([ 0.00000000e+00, -4.99999418e-01,  5.82228685e-07,  5.00000291e-01,
        0.00000000e+00])

In [15]:
pi_mostly_right = np.zeros((len(S), len(A)))
pi_mostly_right[:, 0] = 0.2
pi_mostly_right[:, 1] = 0.8

In [16]:
iterative_policy_evaluation(S, A, R, p, T, pi_mostly_right)

array([0.        , 0.50587989, 0.88235107, 0.97647004, 0.        ])

In [17]:
0.999999 * 0.97647007 * 0.8 + 0.999999 * 0.50588027 * 0.2

0.88235122764789

# Policy Iteration

In [18]:
def policy_iteration(
    model_S: np.ndarray,
    model_A: np.ndarray,
    model_R: np.ndarray,
    model_p: np.ndarray,
    model_T: np.ndarray,
    gamma: float = 0.999999,
    theta: float = 0.000001):
  V = np.random.random(len(model_S))
  V[model_T] = 0.0
  pi = np.zeros((len(model_S), len(model_A)))
  for s_index in range(len(model_S)):
    rdm_a_index = np.random.randint(0, len(model_A))
    pi[s_index, rdm_a_index] = 1.0

  while True:
    # policy evaluation
    while True:
      delta = 0
      for s_index in range(len(model_S)):
        v = V[model_S[s_index]]
        total = 0.0
        for a_index in range(len(model_A)):
          action_total = 0.0
          for s_p_index in range(len(model_S)):
            for r_index in range(len(model_R)):
              action_total += model_p[s_index, a_index, s_p_index, r_index] * (model_R[r_index] + gamma * V[s_p_index])
          total += pi[s_index, a_index] * action_total
        V[model_S[s_index]] = total
        delta = np.maximum(delta, np.abs((total - v)))
      if delta < theta:
        break

    # policy improvement
    policy_stable = True
    for s_index in range(len(model_S)):
      old_a_index = np.argmax(pi[s_index])

      best_a_index = None
      best_a_score = 0.0

      for a_index in range(len(model_A)):
        total = 0
        for s_p_index in range(len(model_S)):
          for r_index in range(len(model_R)):
            total += model_p[s_index, a_index, s_p_index, r_index] * (model_R[r_index] + gamma * V[s_p_index])
        if best_a_index is None or total >= best_a_score:
          best_a_index = a_index
          best_a_score = total

      pi[s_index, :] = 0.0
      pi[s_index, best_a_index] = 1.0

      if old_a_index != best_a_index:
        policy_stable = False

    if policy_stable:
      break

  return pi, V

In [19]:
import time
t = time.time()
pi, v = policy_iteration(
    S, A, R, p, T
)
elapsed = time.time() - t
print(elapsed)
print(pi, v)

0.002715587615966797
[[0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]] [0.       0.999998 0.999999 1.       0.      ]


# Model Free Env Contrat

In [20]:
from typing import List

In [21]:
class ModelFreeEnv:
  def reset(self):
    raise NotImplementedError

  def step(self, action: int):
    raise NotImplementedError

  def is_game_over(self) -> bool:
    raise NotImplementedError

  def current_state(self) -> int:
    raise NotImplementedError

  def available_actions(self) -> List[int]:
    raise NotImplementedError

  def score(self) -> float:
    raise NotImplementedError

  # for performance reasons only
  def maximum_states_count(self) -> int:
    raise NotImplementedError

  def maximum_actions_count(self) -> int:
    raise NotImplementedError

  # optional
  def pretty_print(self):
    raise NotImplementedError



# Line World défini comme un ModelFreeEnv

In [22]:
class LineWorldEnv(ModelFreeEnv):
  def __init__(self, num_cells: int = 5):
    self.num_cells = num_cells
    self.agent_pos = num_cells // 2

  def reset(self):
    self.agent_pos = self.num_cells // 2

  def step(self, action: int):
    if action not in self.available_actions():
      raise Exception("Action is invalid in current state !!!")

    if self.is_game_over():
      raise Exception("Trying to play while game is over !!!")

    if action == 0:
      self.agent_pos -= 1
    elif action == 1:
      self.agent_pos += 1

  def is_game_over(self) -> bool:
    return self.agent_pos == 0 or self.agent_pos == self.num_cells - 1

  def current_state(self) -> int:
    return self.agent_pos

  def available_actions(self) -> List[int]:
    return [0, 1]

  def score(self) -> float:
    if self.agent_pos == 0:
      return -1.0
    elif self.agent_pos == self.num_cells - 1:
      return 1.0
    return 0.0

  # for performance reasons only
  def maximum_states_count(self) -> int:
    return self.num_cells

  def maximum_actions_count(self) -> int:
    return 2

  def pretty_print(self):
    for s in range(self.num_cells):
      if s == self.agent_pos:
        print("X", end="")
      else:
        print("_", end="")
    print()

In [23]:
env = LineWorldEnv()
env.pretty_print()

__X__


In [24]:
env.step(1)
env.pretty_print()

___X_


In [25]:
env.step(1)
env.pretty_print()

____X


In [26]:
env.score()

1.0

In [27]:
env.reset()
env.pretty_print()

__X__


# First Visit MonteCarlo Prediction

In [28]:
def first_visit_monte_carlo_prediction(
    env: ModelFreeEnv,
    pi: np.ndarray,
    iterations_count: int = 10_000,
    gamma: float = 0.999999
) -> np.ndarray:
  V = np.random.random(env.maximum_states_count())
  returns = [[] for s in range(env.maximum_states_count())]

  for it in range(iterations_count):
    env.reset()

    trajectory_states = []
    trajectory_actions = []
    trajectory_rewards = []

    while not env.is_game_over():
      s = env.current_state()
      a_probs = pi[s]
      a = np.random.choice(env.available_actions(), p=a_probs)

      prev_score = env.score()
      env.step(a)
      r = env.score() - prev_score

      trajectory_states.append(s)
      trajectory_actions.append(a)
      trajectory_rewards.append(r)

    V[env.current_state()] = 0.0

    G = 0
    for t, (s, a, r) in reversed(list(enumerate(zip(trajectory_states, trajectory_actions, trajectory_rewards)))):
      G = gamma * G + r
      if s not in trajectory_states[:t]:
        returns[s].append(G)
        V[s] = np.mean(returns[s])
  return V

In [29]:
def optimized_first_visit_monte_carlo_prediction(
    env: ModelFreeEnv,
    pi: np.ndarray,
    iterations_count: int = 10_000,
    gamma: float = 0.999999
) -> np.ndarray:
  V = np.random.random(env.maximum_states_count())
  V_counts = np.zeros(env.maximum_states_count())

  trajectory_states = []
  trajectory_actions = []
  trajectory_rewards = []

  for it in range(iterations_count):
    env.reset()

    trajectory_states.clear()
    trajectory_actions.clear()
    trajectory_rewards.clear()

    while not env.is_game_over():
      s = env.current_state()
      a_probs = pi[s]
      a = np.random.choice(env.available_actions(), p=a_probs)

      prev_score = env.score()
      env.step(a)
      r = env.score() - prev_score

      trajectory_states.append(s)
      trajectory_actions.append(a)
      trajectory_rewards.append(r)

    V[env.current_state()] = 0.0

    G = 0

    t = len(trajectory_states) - 1
    for s, a, r in zip(reversed(trajectory_states), reversed(trajectory_actions), reversed(trajectory_rewards)):
      G = gamma * G + r
      if s not in trajectory_states[:t]:
        V[s] = (V[s] * V_counts[s] + G) / (V_counts[s] + 1)
        V_counts[s] += 1
      t -= 1
  return V

In [30]:
start_time = time.time()
first_visit_monte_carlo_prediction(LineWorldEnv(), pi_right)
print("Elapsed : " + str(time.time() - start_time))

Elapsed : 8.096076011657715


In [31]:
start_time = time.time()
optimized_first_visit_monte_carlo_prediction(LineWorldEnv(), pi_right)
print("Elapsed : " + str(time.time() - start_time))

Elapsed : 0.4897947311401367


In [32]:
first_visit_monte_carlo_prediction(LineWorldEnv(), pi_left)

array([ 0.        , -1.        , -0.999999  ,  0.65123748,  0.68662434])

In [33]:
first_visit_monte_carlo_prediction(LineWorldEnv(), pi_mostly_right)

array([0.        , 0.48973639, 0.8731983 , 0.97678294, 0.        ])

# On Policy First Visit Monte Carlo Control

In [34]:
def optimized_on_policy_first_visit_monte_carlo_control(
    env: ModelFreeEnv,
    iterations_count: int = 10_000,
    gamma: float = 0.999999,
    epsilon: float = 0.05
) -> np.ndarray:
  Q = np.random.random((env.maximum_states_count(), env.maximum_actions_count()))
  Q_counts = np.zeros((env.maximum_states_count(), env.maximum_actions_count()))

  pi = np.ones((env.maximum_states_count(), env.maximum_actions_count())) / env.maximum_actions_count()

  trajectory_states = []
  trajectory_actions = []
  trajectory_rewards = []

  for it in range(iterations_count):
    env.reset()

    trajectory_states.clear()
    trajectory_actions.clear()
    trajectory_rewards.clear()

    while not env.is_game_over():
      s = env.current_state()
      a_probs = pi[s]
      a = np.random.choice(env.available_actions(), p=a_probs)

      prev_score = env.score()
      env.step(a)
      r = env.score() - prev_score

      trajectory_states.append(s)
      trajectory_actions.append(a)
      trajectory_rewards.append(r)

    Q[env.current_state(), :] = 0.0

    G = 0

    t = len(trajectory_states) - 1
    for s, a, r in zip(reversed(trajectory_states), reversed(trajectory_actions), reversed(trajectory_rewards)):
      G = gamma * G + r
      if (s, a) not in zip(trajectory_states[:t], trajectory_actions[:t]):
        Q[s, a] = (Q[s, a] * Q_counts[s, a] + G) / (Q_counts[s, a] + 1)
        Q_counts[s, a] += 1

        best_a = np.argmax(Q[s])
        pi[s, :] = epsilon / env.maximum_actions_count()
        pi[s, best_a] = 1 - epsilon + epsilon / env.maximum_actions_count()

      t -= 1
  return pi, Q

In [35]:
optimized_on_policy_first_visit_monte_carlo_control(LineWorldEnv())

(array([[0.5  , 0.5  ],
        [0.025, 0.975],
        [0.025, 0.975],
        [0.025, 0.975],
        [0.5  , 0.5  ]]),
 array([[ 0.        ,  0.        ],
        [-1.        ,  0.99999794],
        [ 0.96138698,  0.99999895],
        [ 0.99999795,  1.        ],
        [ 0.        ,  0.        ]]))